# firelab — how it works

**D7065E — Embedded Intelligence at the Edge**

firelab is a fire-safety system for a building that exists only in software.
It lights fires in rooms, lets the smoke spread, watches the rooms through
pretend sensors, decides whether there is a real fire, and then acts: it turns
on sprinklers, closes fire doors and sends people to the exits. Everything it
does is drawn in a 3D building viewer called **BuildSim**.

This document walks through the system in the order things happen. Every number
in it comes from the code, and each section names the files it describes, so
you can open them and follow along.

**How to read it**

1. The big picture and what lives where — §1 and §2
2. The loop: what happens every quarter of a second — §3 and §4
3. Each part of the loop in turn: the world, the sensors, the intelligence, the actions, the people — §5 to §9
4. The supporting parts: scoring, the 3D viewer, the control panel, settings — §10 to §16

**Colours** are the same in every diagram:
🟥 the world (what is really happening) · 🟨 sensors (what is measured) ·
🟦 intelligence, the brain (what it thinks and decides) · 🟩 actions (what it does) · ⬜ outside firelab

In [40]:
"""Draws a Mermaid diagram on a dark background so it matches the VS Code dark theme.
Run this cell first. Pass dark=False for a light background."""

import base64
import urllib.request

from IPython.display import HTML, Markdown, display

DARK_BG = "#1e1e2e"
DARK_INIT = (
    "%%{init: {'theme':'dark','themeVariables':{"
    "'background':'#1e1e2e',"
    "'primaryColor':'#2a2a3a','primaryTextColor':'#eeeeee','primaryBorderColor':'#8a8aaa',"
    "'secondaryColor':'#2a2a3a','tertiaryColor':'#2a2a3a',"
    "'lineColor':'#c0c0d0','textColor':'#eeeeee',"
    "'clusterBkg':'#26263a','clusterBorder':'#6a6a8a',"
    "'edgeLabelBackground':'#1e1e2e',"
    "'noteBkgColor':'#3a3a2a','noteTextColor':'#eeeeee',"
    "'actorBkg':'#2a2a3a','actorBorder':'#8a8aaa','actorTextColor':'#eeeeee',"
    "'fontFamily':'Arial, Helvetica, sans-serif','fontSize':'15px'"
    "},"
    "'flowchart':{'padding':22,'nodeSpacing':50,'rankSpacing':70}"
    "}}%%\n"
)

# The drawing service measures text in its own font; the browser may use a wider one.
# So: never clip a label, keep it on one line, and give edge labels their own dark backing.
FIX_CSS = (
    "<style>"
    "#mermaid-svg foreignObject{overflow:visible !important}"
    "#mermaid-svg .nodeLabel,#mermaid-svg .edgeLabel,#mermaid-svg .label{white-space:nowrap !important}"
    "#mermaid-svg .edgeLabel{background:#1e1e2e !important;padding:2px 6px !important;display:inline-block}"
    "#mermaid-svg *{font-family:Arial,Helvetica,sans-serif !important}"
    "</style>"
)

# one colour scheme shared by every diagram (see the colour key above)
STYLE = """
    classDef world fill:#4a2a2a,stroke:#e74c3c,color:#eeeeee
    classDef sense fill:#4a3d1a,stroke:#f1c40f,color:#eeeeee
    classDef brain fill:#1e2e5a,stroke:#5dade2,color:#eeeeee
    classDef act   fill:#1e3a2a,stroke:#58d68d,color:#eeeeee
    classDef ext   fill:#2a2a3a,stroke:#8a8aaa,color:#eeeeee
"""


def mermaid(source: str, timeout: int = 30, dark: bool = True):
    src = source.strip()
    if dark and not src.lstrip().startswith("%%{init"):
        src = DARK_INIT + src
    payload = base64.urlsafe_b64encode(src.encode()).decode()
    bg = f"?bgColor={DARK_BG.lstrip(chr(35))}" if dark else ""
    url = f"https://mermaid.ink/svg/{payload}{bg}"
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            svg = response.read().decode()
            if dark:
                html = (
                    f'<div style="background:{DARK_BG};padding:16px;'
                    f'border-radius:8px;display:inline-block;width:100%;'
                    f'box-sizing:border-box">{FIX_CSS}{svg}</div>'
                )
                display(HTML(html))
            else:
                from IPython.display import SVG as _SVG
                display(_SVG(svg))
    except Exception as exc:
        display(Markdown(f"> Diagram not rendered ({exc}).\n\n```mermaid\n{source.strip()}\n```"))

## 1. The big picture

Two programs run on your computer:

| | What it is | What it does |
|---|---|---|
| **BuildSim** | a Go program on port 9090 (`../buildingsim`) | Knows the building: the floor plans, which rooms connect to which, where the stairs are. Draws the 3D view. It does **not** simulate anything. |
| **firelab** | a Python program on port 8090 (this folder) | Everything else: fire and smoke, sensors, decisions, sprinklers, doors, evacuation, scoring. It also serves its own control panel. |

You keep two browser tabs open. In the first you drive the run from firelab's
control panel. In the second you watch the building react in BuildSim's viewer.

In [39]:
BIG_PICTURE = """
flowchart LR
    YOU["You"]

    subgraph FL["firelab · port 8090"]
        UI["Control panel"]
        ENG["Engine<br/>fire, sensors, decisions,<br/>evacuation, scoring"]
    end

    subgraph BS["BuildSim · port 9090"]
        VIEW["3D viewer"]
        DATA["The building<br/>floor plans, rooms,<br/>stairs, escape routes"]
    end

    YOU -->|"tab 1: drive the run"| UI
    UI <-->|"POST buttons in,<br/>SSE live updates out"| ENG
    ENG -->|"PUT fire, smoke, alarms,<br/>doors, people"| VIEW
    DATA -->|"GET floor plans,<br/>stairs, escape routes"| ENG
    VIEW -->|"tab 2: watch"| YOU

    %% invisible links: keep You on the left and BuildSim on the right
    YOU ~~~ VIEW
    ENG ~~~ DATA

    class YOU,VIEW,DATA ext
    class UI,ENG act
"""

mermaid(BIG_PICTURE + STYLE)

**How to read the arrows.** An arrow points the way the data travels. `PUT`
sends drawing instructions to BuildSim, so it points right; `GET` brings the
floor plans and routes back, so it points left; and "watch" brings the picture
to your eyes. Even so, **firelab starts every exchange**: it reads the floor
plans and stairs once at startup, asks for an escape route whenever someone
needs one, and `PUT`s what to draw about once a second. BuildSim only ever
answers; it never sends anything on its own, and firelab does not poll it for
status beyond one `GET /healthz` at startup and after **Reload**. If a call
fails, firelab marks itself *disconnected* and stops drawing until you press
Reload. firelab never writes to the building: the floor plans are BuildSim's
and are only ever read.

firelab never changes BuildSim's code. It only uses BuildSim's web API, and
every one of those calls lives in `adapters/buildsim/client.py` and `viewer.py`.

**Running it**

```bash
cd buildingsim && make build && make run                               # terminal 1 → http://127.0.0.1:9090
cd firelab && make build && make run                     # terminal 2 → http://127.0.0.1:8090
make test                                                # 97 tests, nothing needs to be running
```

## 2. Inside firelab

firelab is split into a few folders, each with one job. The arrows below show
who is allowed to talk to whom.

In [38]:
INSIDE = """
flowchart LR
    YOU["You"]

    subgraph FL["firelab"]
        UI["ui/<br/>control panel"]
        API["app/api/<br/>web endpoints"]
        ENG["app/engine/<br/>the loop"]
        DOM["domain/<br/>the rules: physics, sensors,<br/>detector, safety, people"]
        ADP["adapters/<br/>talks to BuildSim"]
    end

    BS[("BuildSim")]

    YOU --> UI
    UI -->|"POST / PUT commands"| API
    API -->|"SSE snapshot<br/>every tick"| UI
    API <-->|"calls the engine,<br/>reads its state"| ENG
    ENG <-->|"hands in rooms and readings,<br/>gets results back"| DOM
    ENG <-->|"hands in state,<br/>gets BuildSim's answers back"| ADP
    ADP -->|"PUT layers, effects, alerts,<br/>doors, people, sensor values"| BS
    BS -->|"GET floor plans, stairs,<br/>escape routes"| ADP
    BS -->|"watch"| YOU

    %% invisible links: keep You on the left and BuildSim on the right
    YOU ~~~ BS
    ADP ~~~ BS

    class YOU,BS ext
    class UI,API,ENG act
    class ADP sense
    class DOM brain
"""

mermaid(INSIDE + STYLE)

| Folder | Job | The one rule it follows |
|---|---|---|
| `ui/` | the control panel (Svelte 5 + Vite, built into `ui/dist`) | only talks to `app/api/`; never works anything out itself |
| `app/` | the running program: the loop (`engine/`), the web endpoints (`api/`), the snapshot the panel shows (`snapshot/`), evacuation, settings, presets | owns all the changing state and the clock |
| `adapters/` | the bridge to BuildSim: the HTTP client, turning floor plans into rooms (`world_builder.py`), turning engine state into things BuildSim can draw (`publisher/`) | the **only** code that talks over the network |
| `domain/` | the pure rules: fires, physics, sensors, features, detector, alarm logic, safety checks, people, scoring | uses only the Python standard library; no network, no clock, no settings. Time and randomness are passed in |

The two-headed arrows inside firelab are function calls: data goes in, a result
comes back. What *is* one-way is who may import whom: `app` → `adapters` →
`domain`, never the other way round. Because of this,
every rule in `domain/` can be tested on its own with nothing running, and the
whole test suite finishes in about 1.6 seconds. `tests/test_layering.py` reads
the source files and fails if any of these rules is broken.

**Folder map**

```
firelab/
  ui/                    control panel; src/lib has one component per panel
  app/
    main.py                starts the web server and the engine
    runtime.py             the single Engine instance everyone shares
    config.py              settings you can change while it runs
    presets.py             six one-click scenarios
    evacuation.py          asks BuildSim for routes and walks people along them
    engine/                state, the tick loop, and one file per step of the loop
    snapshot/              turns engine state into the JSON the panel shows
    api/                   the web endpoints, one file per group
  adapters/
    buildsim/              HTTP client for BuildSim
    world_builder.py       floor plans + stairs → rooms and doorways
    publisher/             engine state → heat maps, effects, alerts, doors, people
  domain/
    world.py               rooms and doorways
    sources.py             what a fire (or a toaster) gives off over time
    physics.py             how heat, smoke and CO settle and spread
    sensing.py             what a sensor does to the true value
    features.py            summarises the last 120 s of readings
    detector.py            turns features into P(fire)
    agent.py               the alarm ladder and the commands it issues
    interlocks.py          the safety checks that can refuse a command
    roles.py, occupants.py people: who they are, where they start, how they walk
    tenability.py          when a room becomes unsafe to be in
    scoring.py, timeline.py, history.py   grading the run
  tests/                   15 test files
```

## 3. The loop

The whole system is one feedback loop with four zones. Read it left to right,
then follow the thick arrow back to the start.

In [37]:
LOOP = """
flowchart LR
    W["1 · WORLD<br/>fires burn, smoke spreads<br/>what is really happening"]
    S["2 · SENSORS<br/>smoke, CO, temperature<br/>late, noisy, can break"]
    I["3 · INTELLIGENCE<br/>the brain: is it a fire?<br/>what should we do?"]
    A["4 · ACTIONS<br/>safety check, then<br/>sprinkler, door, evacuate"]

    W -->|"true values"| S
    S -->|"readings"| I
    I -->|"commands"| A
    A ==>|"feedback: a sprinkler cools the room,<br/>a closed door slows the smoke"| W

    class W world
    class S sense
    class I brain
    class A act
"""

mermaid(LOOP + STYLE)

| Zone | What it is allowed to look at | What it produces | Where in the code |
|---|---|---|---|
| **World** | the fires, the previous state, sprinklers and doors | the true temperature, smoke and CO in every room | `engine/truth.py` → `domain/physics.py` |
| **Sensors** | the true value of their own room | readings that are late, noisy and sometimes wrong | `engine/truth.py` → `domain/sensing.py` |
| **Intelligence** (the brain) | readings only — **never the truth** | P(fire) per room, an alarm level, commands | `engine/intelligence.py` |
| **Actions** | the commands, plus the few facts the safety checks need | sprinkler on/off, door open/closed, evacuation | `engine/actuation.py` |

Two ideas make this a real control system rather than a dashboard:

1. **The intelligence cannot peek.** The detector only sees what the sensors report. If
   it could read the true smoke level it would be perfect, and false alarms,
   missed fires and broken sensors would mean nothing. A test fails if
   `engine/intelligence.py` ever reads `.temperature`, `.smoke` or `.co`.
2. **Actions change the world.** A sprinkler cuts the room's heat to 25 % and its
   smoke to 40 % on the next physics step. A closed fire door shrinks every
   doorway into the room to 15 % open, so smoke leaks through more slowly. What
   the system does changes what its sensors read next.

## 4. Startup, then every quarter of a second

### 4.1 Startup

When firelab starts (`app/main.py` → `engine.start()` → `engine/building.py`)
it fetches the building from BuildSim once and builds its own copy.

In [41]:
STARTUP = """
sequenceDiagram
    participant F as firelab
    participant B as BuildSim
    F->>B: GET /healthz — are you there?
    B-->>F: yes
    loop level0, level1, level2
        F->>B: GET the floor plan for this level
        B-->>F: rooms and walkable links
    end
    Note over F: every named room becomes a Space<br/>every walkable link between two rooms becomes a doorway
    F->>B: GET the stairs between floors
    B-->>F: the stairwells
    Note over F: each stairwell becomes a doorway between floors<br/>a copy of the walkable map is kept for finding clear escape routes<br/>people are scattered into the rooms
    Note over F: the loop starts
"""

mermaid(STARTUP)

If BuildSim is not running, firelab stays up but marks itself *not connected*
and waits until you press **Reload** (`POST /api/reload`). If one floor fails
to load, the others still work. If the stairs fail to load, each floor works on
its own, but smoke and people cannot move between floors.

### 4.2 One tick

The loop (`engine/core.py`) wakes up every **0.25 s** of real time. If the
building is loaded and the clock is running, it does the following, in order:

In [46]:
TICK = """
flowchart TB
    T(["every 0.25 s of real time<br/>simulated time = real time × speed factor"])
    subgraph SUB["repeated in small steps of about 1 simulated second"]
        direction LR
        W["World<br/>fires burn, smoke spreads"]
        S["Sensors<br/>take readings"]
        E["People<br/>breathe in CO"]
        W -->|"true smoke, CO, heat"| S
        W -->|"true CO"| E
    end
    B["Intelligence<br/>is it a fire? what to do?"]
    A["Actions<br/>safety check, then act"]
    R["Record<br/>truth next to reading, every 5 s"]
    P["People<br/>find a route and walk"]
    D{"a second since<br/>the last drawing?"}
    BS["Draw in BuildSim"]
    UI["Send a snapshot to the control panel"]

    T --> SUB --> B --> A --> R --> P --> D
    D -->|"yes"| BS --> UI
    D -->|"no"| UI
    UI -.->|"next tick"| T

    class W,E world
    class S sense
    class B brain
    class A,P act
    class BS,UI ext
"""

mermaid(TICK + STYLE)

Inside the box, the arrows show what flows: the sensors read the world's true
values, and people breathe the world's true CO. Nothing passes from the sensors
to the people; both take from the world. Outside the box, the arrows are the
order the steps run in.

A few details worth knowing:

- **Small steps.** If you fast-forward, one tick may cover many simulated
  seconds. The world and sensor steps are repeated in steps of about one
  simulated second (at most 60 per tick) so the physics stays stable. The
  intelligence runs once per tick.
- **Drawing is rate-limited** to about once per real second, so BuildSim is
  never flooded.
- **While paused**, nothing moves, but if you change something (deploy a sensor,
  press a button, change the population) it is still drawn in BuildSim.
- **The clock** starts at 08:00:00 and only moves while the run is playing.

## 5. The world: rooms, doorways and fires

*Code: `domain/world.py`, `domain/sources.py`, `domain/physics.py`, `adapters/world_builder.py`*

### 5.1 Rooms and doorways

firelab's model of the building is deliberately simple:

| Thing | What it is | Where it comes from |
|---|---|---|
| **Space** (a room) | one node with a temperature, a smoke level, a CO level and a sprinkler flag | every named room in a BuildSim floor plan. The plan is in half-metre units, so area = plan area × 0.25 m², and volume = area × 3 m (never less than 10 m³) |
| **Coupling** (a doorway) | a link between two rooms with an *openness* from 0 (sealed) to 1 (wide open) | every walkable link in the floor plan whose two ends are in **different** rooms. A stairwell becomes a doorway between floors with a fixed strength of 0.6 |

A room that has no walkable point in the plan is marked *not routable*, and no
people are placed there. Reset returns every room to 20 °C, no smoke, no CO,
sprinklers off and every door open.

### 5.2 Fires and things that look like fires

A **source** is placed in a room at a chosen time. Its *kind* is the ground
truth: two kinds are real fires, three are nuisances that a good detector should
ignore. Each kind gives off heat, smoke and CO in its own pattern
(`e` is the number of seconds since it started):

| Kind | Real fire? | Heat (kW) | Smoke (1/m) | CO (ppm) | In plain words |
|---|---|---|---|---|---|
| `flaming` | yes | `Q = min(α·e², peak_kw)` | `0.06·√Q` | `4·√Q` | grows faster and faster until it hits its cap |
| `smouldering` | yes | `3·r` | `0.9·r` | `900·r` | `r = 1 − e^(−e/300)`: thick smoke, lots of CO, almost no heat |
| `cooking` | no | `12·b` | `0.35·b` | `25·b` | warm and smoky for a while, then fades |
| `dust` | no | 0 | `0.5·b` | 0 | smoke only, brief |
| `steam` | no | 0 | `0.7·b` | 0 | smoke only, brief |

`α` sets how fast a flaming fire grows: slow 0.0029, medium 0.0117, fast 0.0469,
ultrafast 0.1876 kW/s². `b` is a bump that rises, holds and then dies away.
Several sources in one room simply add up.

### 5.3 How heat and smoke move

Every small step, physics does two things:

**1. Each room settles towards its own target.** A room with `Q` kW of heat and
volume `V` heads for

$$T_{target} = 20 + \min\!\left(140\cdot\left(\tfrac{Q}{V}\right)^{2/3},\ 900\right)\ °C$$

and moves towards it gradually: temperature with a time constant of 90 s, smoke
45 s, CO 60 s. If the sprinkler is on, the heat is first cut to 25 % and the
smoke to 40 %.

**2. Neighbouring rooms swap through their doorways.** For each doorway the
share that crosses depends on how wide it is, how open it is and the quantity
(temperature spreads slowest, smoke fastest). Two safety rules keep it honest:
no doorway ever moves more than half the difference in one step, and a room
with many doors (a corridor) never gives away more than half its difference in
total, so smoke is never created out of nothing. All the swaps are worked out
first and applied afterwards, so the order of the doors does not matter.

Closing a fire door sets every doorway into that room to 15 % open. Opening it
sets them back to 100 %.

## 6. The sensors

*Code: `domain/sensing.py`, `app/engine/devices.py`, `app/engine/truth.py`*

### 6.1 Putting sensors in a room

A room is only **monitored** once you deploy sensors in it. `deploy()` creates
one device per room and per modality (smoke, CO, temperature), with an id like
`fl-level0-A1016-smoke`, and gives that room a reading window and an alarm
agent. The devices are also registered in BuildSim's equipment tree so they
show up in the viewer.

A room with no sensors has no readings, no P(fire) and no agent, so it can never
raise an alarm, no matter how hard it burns. That is on purpose: it is how you
test what happens with a blind spot.

### 6.2 What a reading goes through

A real sensor never reports the truth exactly. Every small step, each device:

1. **lags** behind the true value (it takes a while to warm up or fill with smoke);
2. only **reports** when its sampling interval is due (5 s by default);
3. applies its **fault**, if it has one;
4. adds its **bias and drift**;
5. adds random **noise**;
6. **rounds** to what the hardware can resolve;
7. **clamps** to the range the hardware can measure.

| Sensor | Lag (s) | Noise | Rounds to | Range |
|---|---|---|---|---|
| smoke (1/m) | 12 | 0.010 | 0.005 | 0 – 3 |
| CO (ppm) | 25 | 3.0 | 1 | 0 – 2000 |
| temperature (°C) | 30 | 0.15 | 0.1 | −20 – 250 |

You can break a sensor from the panel (`POST /api/sensors/fault`) or with a preset:

| Fault | What it does |
|---|---|
| `dead` | never reports again |
| `dropout` | skips 40 % of its readings |
| `stuck` | keeps repeating its last value |
| `drift` | its error keeps growing a little with every reading |

All randomness comes from one seeded random generator, so a run with the same
seed and the same clicks repeats exactly.

## 7. The intelligence — the brain of the system

*Code: `domain/features.py`, `domain/detector.py`, `domain/agent.py`, `app/engine/intelligence.py`*

This is the "intelligence" in *Embedded Intelligence at the Edge*. It lives in
four files: `app/engine/intelligence.py` runs it once per tick, and the rules it
runs are `domain/features.py` (summarise the readings), `domain/detector.py`
(turn them into P(fire)) and `domain/agent.py` (climb the alarm ladder and
issue commands). Everything else in firelab exists to feed it or to act on it.

Once per tick, for every monitored room, it turns the recent readings into a
probability and then into an alarm level. It only ever sees readings, never the
true state of the room; that boundary is what makes it a detector rather than
a cheat, and a test fails if it is crossed.

In [47]:
BRAIN = """
flowchart LR
    WIN["Last 120 s of readings<br/>for this room"]
    NB["Latest smoke reading<br/>in the neighbouring rooms"]
    F["Features<br/>smoke, CO, heat rise, smoke rate,<br/>CO-to-smoke ratio, neighbour agreement,<br/>how many sensor types have reported"]
    P["P(fire)<br/>a number from 0 to 1"]
    L["Alarm ladder<br/>NORMAL → … → CONFIRMED"]
    C["Commands<br/>sprinkler, door, evacuate"]
    SC["Scoreboard<br/>was that alarm right? how late?"]

    WIN --> F
    NB --> F
    F --> P --> L --> C
    L --> SC

    class WIN,NB sense
    class F,P,L,C,SC brain
"""

mermaid(BRAIN + STYLE)

### 7.1 Features

Each room keeps the last **120 s** of readings. From them it works out:

| Feature | Meaning |
|---|---|
| `smoke`, `co`, `temperature` | the latest reading of each (temperature counts as 20 °C if there is none) |
| `temperature_rise` | how far above 20 °C the room is |
| `smoke_rate` | how fast the smoke reading is climbing, per minute |
| `co_smoke_ratio` | CO divided by smoke — high for a smoulder, low for dust or steam |
| `neighbour_agreement` | what fraction of the monitored rooms next door also see smoke |
| `coverage` | how many of the three sensor types have reported at all (0 to 3) |

### 7.2 From features to P(fire)

The detector is a hand-written rule called `FusionRule`. It gives each feature a
weight, adds them up with a starting bias of −3.4, and squashes the total into a
probability:

| Term | Contribution |
|---|---|
| bias | −3.4 |
| smoke | 3.2 × min(smoke, 1.5) |
| CO | 1.8 × clamp((co − 25) / 125, 0, 1) |
| CO-to-smoke ratio | 1.6 × min(ratio / 250, 2.5) × min(smoke / 0.15, 1) |
| temperature rise | 0.55 × min(rise, 60) / 10 |
| smoke rate | 1.4 × clamp(rate, 0, 1) |
| neighbour agreement | 0.8 × agreement |

$$P(\text{fire}) = \text{logistic}\Big(\sum \text{terms}\Big) \times \Big(0.55 + 0.45\cdot\tfrac{\text{coverage}}{3}\Big)$$

The last factor matters: with **no** working sensor P is 0, and with only
**one** working sensor P can never go above 0.70. The confirm threshold is 0.75,
so **one sensor on its own can never confirm a fire**. The "Why" panel in the
control panel shows exactly these terms.

Today the intelligence is a hand-written rule, not a learned model. The
detector is the one thing designed to be swapped out. `Detector` is a small
interface (`name`, `probability`, `explain`), and `FusionRule()` is installed in
`EngineState.__init__`. A trained model would go in the same place, and the CSV
export (§10) gives you labelled rows to train it on.

### 7.3 The alarm ladder

Each monitored room has an **agent** that climbs and descends a ladder of alarm
levels. It moves at most one step per tick, and most steps have to be *held* for
a while so a brief puff of smoke never sets off the sprinklers.

In [48]:
LADDER = """
stateDiagram-v2
    direction LR
    [*] --> NORMAL
    NORMAL --> INVESTIGATING: P ≥ 0.35
    INVESTIGATING --> NORMAL: P < 0.25
    INVESTIGATING --> PRE_ALARM: P ≥ 0.55 held for 20 s
    PRE_ALARM --> INVESTIGATING: P < 0.35
    PRE_ALARM --> CONFIRMED: P ≥ 0.75 after 25 s in PRE_ALARM
    CONFIRMED --> SUPPRESSED: sprinkler on and P < 0.75 for 25 s
    SUPPRESSED --> CONFIRMED: P ≥ 0.75
    CONFIRMED --> CLEARING: P ≤ 0.25 for 90 s
    SUPPRESSED --> CLEARING: P ≤ 0.25 for 90 s
    CLEARING --> INVESTIGATING: P ≥ 0.35
    CLEARING --> NORMAL: 90 s in CLEARING
"""

mermaid(LADDER)

The four thresholds (0.35, 0.55, 0.75, 0.25) and the three hold times (20 s,
25 s, 90 s) live in the settings and can be changed while it runs.

Commands are issued when a room **enters** a level:

| Entering | Commands |
|---|---|
| `CONFIRMED` | sprinkler **on**, fire door **closed**, evacuation **start** |
| `CLEARING` | sprinkler **off**, fire door **open**, evacuation **stop** |
| `SUPPRESSED` | nothing new; the CONFIRMED commands stay on the list |
| anything else | the list is cleared |

A command stays on the agent's list, and is offered again **every tick**, until
it has actually been carried out. So a command that the safety checks refuse
(for example, a sprinkler in a room that is not yet hot) is delayed, not lost.

In **supervised mode** (`auto = false`) the agent still climbs the ladder, but its
commands are not carried out automatically. They wait until an operator presses
the button.

## 8. Safety checks and actions

*Code: `domain/interlocks.py`, `app/engine/actuation.py`*

`actuation.apply()` is the only place where anything physical changes. Commands
from the agent and clicks from the operator both go through it, so a button can
never skip a safety check.

In [49]:
ACTIONS = """
flowchart LR
    AG["Agent's commands"]
    OP["Operator's button"]
    MODE{"automatic<br/>mode?"}
    WAIT["Waits for the operator"]
    CHK["Safety checks<br/>is the room hot enough?<br/>is anyone inside?<br/>is it on an escape route?"]
    NO["Refused<br/>written in the journal,<br/>offered again next tick"]
    YES["Carried out<br/>sprinkler on or off<br/>door open or closed<br/>everyone starts leaving"]

    AG --> MODE
    MODE -->|"yes"| CHK
    MODE -->|"no"| WAIT
    OP --> CHK
    CHK -->|"refused"| NO
    CHK -->|"allowed"| YES

    class AG brain
    class OP,WAIT,NO ext
    class CHK,YES act
"""

mermaid(ACTIONS + STYLE)

### 8.1 The safety checks

| Command | Refused when |
|---|---|
| sprinkler **on** | the room's true temperature is below **35 °C** — smoke alone never releases water, just like a real heat-activated sprinkler head |
| fire door **locked** | always — fire doors must never lock people in |
| fire door **closed** | anyone is still in the room, or the room is on someone's escape route |
| everything else | never (sprinkler off, door open, evacuation start or stop) |

### 8.2 What an allowed command does

| Command | Effect |
|---|---|
| sprinkler on / off | the room's sprinkler flag flips; physics reads it on the next step |
| fire door closed / open | every doorway into the room goes to 15 % / 100 % open |
| evacuation start | **everyone** still in the building starts leaving — one confirmed room empties the whole building |
| evacuation stop | everyone goes back to idle, but only if no room is still in danger |

A refusal is written to the journal only when the reason changes, so a command
retried every tick does not flood the log.

## 9. People and getting out

*Code: `domain/roles.py`, `domain/occupants.py`, `app/evacuation.py`*

### 9.1 Who is in the building

| Role | How many by default | Walking speed | Note |
|---|---|---|---|
| student | 28 | × 1.05 | quick, but only know the main entrance |
| lecturer | 5 | × 1.00 | know the building |
| staff | 4 | × 1.00 | technical and administrative staff |
| security | 1 | × 1.15 | drilled on the escape routes |
| visitor | 2 | × 0.85 | have never been here before |

At startup people are scattered at random over rooms that are routable, are not
corridors, and are bigger than 8 m². You can change the mix from the panel (up to
2000 people). The base walking speed is 1.3 m/s.

### 9.2 Finding the way out

A room is **in danger** when its alarm level is PRE_ALARM, CONFIRMED or
SUPPRESSED. While any room is in danger, this runs every tick for every person:

In [50]:
ESCAPE = """
flowchart TB
    Q1{"Is this person in a danger room,<br/>or already leaving?"}
    STAY["Stays put"]
    Q2{"Already has a route<br/>that avoids the danger?"}
    WALK["Keeps walking<br/>1.3 m/s × role speed"]
    ASK["Looks for a way to an exit that is not in danger:<br/>first a clear route on firelab's own map,<br/>then BuildSim's shortest route to each exit"]
    PICK["A clear route wins; otherwise the one that<br/>crosses the fewest danger rooms, then the shortest"]
    NONE["Marked as: no route"]
    SAFE(["Safe when the last<br/>waypoint is reached"])

    Q1 -->|"no"| STAY
    Q1 -->|"yes"| Q2
    Q2 -->|"yes"| WALK
    Q2 -->|"no"| ASK --> PICK
    PICK -->|"found one, or the current<br/>route is still the best"| WALK
    PICK -->|"no route to any exit"| NONE
    WALK --> SAFE

    class Q1,Q2 brain
    class ASK,PICK,WALK act
    class STAY,NONE,SAFE ext
"""

mermaid(ESCAPE + STYLE)

- **Exits** are the real ground-floor doors listed in the settings
  (`A1016, A1123, A105, A10, A1000A, A1000E, A170` on `level0`). People upstairs
  are routed down the stairs to one of them. An exit that is itself in danger is
  never used.
- **Ranking.** A clear route always wins, but a smoky route is still better than
  none, because one corridor is often the only way out of a wing. Between routes
  with the same number of danger rooms, the shorter wins, and a change of floor
  counts as 20 extra units.
- **No jogging on the spot.** Someone who is already walking only switches to a
  new route if it crosses *fewer* danger rooms than what is left of their current
  one.
- **Caching.** The best route from each room is remembered until the set of
  danger rooms changes.
- **Stranded.** "No route" means BuildSim found no path to any exit. These people
  are listed as stranded in the panel.
- **BuildSim is told too.** Danger rooms are sent to BuildSim as *blocked*, so its
  own router avoids them where it can. Corridors cannot be blocked that way, which
  is why firelab does its own ranking as well.
- One route is drawn in the viewer: the one that starts closest to the fire.

## 10. Keeping score

*Code: `domain/tenability.py`, `domain/scoring.py`, `domain/timeline.py`, `domain/history.py`*

Because every source carries its true kind, firelab can grade itself while the
run is happening.

### 10.1 Is the room still safe to be in?

Following ISO 13571, a room is **untenable** when its true temperature reaches
**60 °C** or its smoke reaches **0.30 /m** (you can see about 10 m). Everyone
still inside also breathes in CO every small step, building up a dose
(`fed += co^1.036 / 35000 × dt / 60`). A dose of **0.3** or more counts as
**incapacitated**, even if the person reaches an exit later.

### 10.2 Was the alarm right?

Every new alarm (a room entering CONFIRMED or SUPPRESSED) is judged once:

| Outcome | When |
|---|---|
| **detection** | a real fire is burning in that room; the latency is the time since it started |
| **excused** | no fire in this room, but a real fire next door — the smoke came through the doorway |
| **false alarm** | anything else; the cause is the nuisance in the room, if any |
| **miss** | a real fire has burned for more than **300 s** without being detected |

### 10.3 How long did it take?

| Metric | Measured from → to |
|---|---|
| `alarm_delay` | first real fire → first alarm |
| `rset` | first real fire → building empty |
| `rset_from_alarm` | first alarm → building empty |

### 10.4 The record, and the CSV

Every **5 simulated seconds**, one row is stored per monitored room with the
true values, the readings and P(fire) side by side. Each room keeps its last
240 rows (about 20 minutes). This is the only place truth and readings are
deliberately stored together.

- `GET /api/history?space=` returns one room's rows plus the detector's "why" terms. The panel's charts use it.
- `GET /api/export.csv` returns the whole run with a `label` column: the kind of
  source that was in that room, or `none`. This is ready for training a detector offline.

## 11. Drawing it in BuildSim

*Code: `app/engine/publishing.py`, `adapters/publisher/`*

About once per real second, the engine turns its state into things BuildSim can
draw and sends them all at once:

| What you see in the viewer | BuildSim endpoint | Notes |
|---|---|---|
| four floor heat maps: true temperature, true smoke, true CO, and **P(fire)** | `PUT /api/room-layers` | truth and belief side by side, so you can compare them |
| fire, smoke and sprinkler effects | `PUT /api/effects` | fire where a source is burning; smoke in the 40 smokiest rooms |
| alerts | `PUT /api/alerts` | one per room that is not NORMAL |
| fire doors | `PUT /api/doors` | commanded and danger rooms; never locked; danger rooms are *blocked* |
| people walking | `PUT /api/entities`, `PUT /api/occupancy` | up to 2000, with a smooth glide between updates |
| coloured room outlines | `PUT /api/sessions/{id}/highlights` | red confirmed, orange pre-alarm, yellow investigating, green clearing; only sent when changed |
| the escape route | `PUT /api/sessions/{id}/route` | only sent when changed |
| sensor values | `PUT /api/sensors/{id}/value` | at most 24 per second |
| sensor devices | `POST /api/equipment/bulk` | once, when you deploy |

The outlines and the route are drawn into a **viewer session**: the most
recently active BuildSim tab, looked up every 30 s. If any call fails, the
engine marks itself disconnected and stops drawing until you press **Reload**.

## 12. The control panel

*Code: `ui/`, `app/api/`, `app/snapshot/`*

### 12.1 How the panel stays up to date

The panel never works anything out itself. Once a tick, the engine builds one
**snapshot** (`app/snapshot/builder.py`) and pushes it to every open panel over
a Server-Sent Events stream (`GET /api/events`). The snapshot holds the clock,
the connection status, one row per room being watched, the sources, the
counts, the population, tenability, the score, the evacuation summary and the
last 40 journal lines. If a browser falls behind, its oldest snapshot is
dropped, so a slow tab never slows the engine.

The panel (`ui/src/App.svelte`) has three columns:

| Left | Centre | Right |
|---|---|---|
| Scenario, Scorecard, Evacuation, Population, Sensors | Room table, Room detail (charts + Why) | Response, Tenability, Settings, Journal |

### 12.2 The endpoints

Every button in the panel calls one of these:

| Group | Endpoint | Does |
|---|---|---|
| **Run** | `POST /api/clock` | play, pause, set the speed factor (0.1 – 600) or the time |
| | `POST /api/reset` | start the run again (§14) |
| **Scenario** | `POST /api/scenario/ignite` | place a source: room, kind, growth, delay, peak kW |
| | `POST /api/scenario/preset` | run one of the six presets (§13) |
| | `DELETE /api/scenario/{id}` | remove a source |
| **Sensors** | `POST /api/sensors/deploy` | put sensors in rooms |
| | `POST /api/sensors/undeploy` | take them out |
| | `POST /api/sensors/fault` | break one |
| **People** | `PUT /api/population` | change the mix and re-scatter |
| **Response** | `POST /api/agent/mode` | automatic or supervised |
| | `POST /api/actuators/command` | press a sprinkler / door / evacuate button (still safety-checked) |
| **Data** | `GET /api/history?space=` | one room's record and "why" |
| | `GET /api/export.csv` | the whole run, labelled |
| **System** | `GET /api/state`, `GET /api/events` | the snapshot, once or as a stream |
| | `GET /api/rooms`, `GET /api/roles`, `GET /api/presets` | lists for the pickers |
| | `GET` / `PUT /api/config` | read or change settings |
| | `POST /api/reload`, `GET /healthz` | reconnect to BuildSim; health |

`GET /` serves the built panel from `ui/dist`.

## 13. Settings and presets

*Code: `app/config.py`, `app/presets.py`*

### 13.1 Settings

Everything below can be changed while it runs with `PUT /api/config`:

| Setting | Default |
|---|---|
| `buildsim_url` | `http://127.0.0.1:9090` |
| `levels` | `level0, level1, level2` |
| `factor` | 1.0 simulated second per real second |
| `step_seconds` | 1.0 |
| `seed` | 1 (a new seed restarts the random generator) |
| `population` | the counts in §9.1 |
| `walking_speed` | 1.3 m/s |
| `sensors` | interval 5 s, noise scale 1.0, all three modalities |
| `response` | automatic mode on, and the thresholds in §7.3 |
| `exits` | the seven ground-floor doors in §9.2 |

The sensor interval and noise scale apply to sensors deployed **after** the change.

### 13.2 Presets

A preset deploys all three sensors in a room **and its neighbours** (so
neighbour agreement has something to work with), breaks the smoke sensor if the
preset says so, and lights the source:

| Preset | Source | Broken sensor | What should happen |
|---|---|---|---|
| `office-fire` | flaming, fast | — | detected quickly |
| `overnight-smoulder` | smouldering | — | detected on CO; sprinkler refused because the room is not hot |
| `kitchen-cooking` | cooking | — | no alarm |
| `dust-storm` | dust | — | no alarm |
| `blind-spot` | flaming, fast | smoke sensor dead | still detected on CO and heat, but later |
| `drifting-sensor` | none | smoke sensor drifting | no alarm |

`tests/test_presets.py` runs every preset from start to finish and checks the last column.

## 14. Reset, reload, and where the state lives

*Code: `app/engine/state.py`, `app/engine/session.py`, `app/runtime.py`*

All changing data lives in one object, `EngineState`: the world, the sources,
the sensors, the readings, the probabilities, the agents, the people, the doors,
the journal (last 200 lines), the score, the history and the random generator.
`app/runtime.py` creates the single `Engine` that every endpoint shares.

Everything runs on one asyncio event loop. A tick holds the engine lock while it
runs. Endpoints do not take the lock, but most of them change state without
waiting on anything, so they cannot land in the middle of a physics step.

**Reset** and **Reload** are different buttons:

| | Reset (`POST /api/reset`) | Reload (`POST /api/reload`) |
|---|---|---|
| the building | kept | fetched again from BuildSim and rebuilt |
| sensors | kept, but readings and **faults cleared** | kept |
| heat, smoke, doors, sprinklers | cleared | fresh |
| sources, probabilities, agents | cleared, agents back to NORMAL | unchanged |
| score, history | cleared | unchanged |
| journal, clock | kept | kept |
| people | re-scattered | re-scattered |
| settings | kept | kept |

## 15. Tests

`make test` runs **97 tests** (plus 82 sub-tests) in about 1.6 s. None of them
need BuildSim; the evacuation tests use a fake client.

| File | Checks |
|---|---|
| `test_layering.py` | the folder rules in §2 |
| `test_sources.py` | fires grow and cap; smoulder looks like a smoulder; nuisances give off no CO |
| `test_physics.py` | smoke spreads next door; sprinklers suppress; smoke is never created; results do not depend on the clock speed |
| `test_multifloor.py` | stairs join the floors; smoke climbs them; routes cross them |
| `test_sensing.py` | sensors lag then track; dead and stuck faults |
| `test_detector.py` | quiet room stays quiet; no sensors means P = 0; fire beats dust; smoulder is caught on CO; cooking stays below pre-alarm |
| `test_agent.py` | the ladder climbs; brief blips never confirm; commands are re-offered; suppressed; back to normal |
| `test_interlocks.py` | smoke alone never releases water; doors never lock; no sealing occupied rooms or routes |
| `test_publisher_doors.py` | blocked flags; never locked; stable order |
| `test_occupants.py` | placement; walking to safety |
| `test_evacuation.py` | route choice, ranking, caching, re-routing, stranded people, display route |
| `test_scoring.py` | latency, false alarms, misses, the neighbour excuse |
| `test_history.py` | rows every 5 s and the 240 limit |
| `test_presets.py` | every preset from ignition to actuation |
| `test_reset.py` | what reset clears and what it keeps |

## 16. What it does not do

These are honest facts about the code today, not plans:

- **The detector is a hand-written rule**, not a trained model. Swapping in a model
  is a one-line change (§7.2), and the CSV export gives you labelled data.
- **It does not predict** where the fire will go next. It reacts to what it sees now.
- **The physics is simple:** one well-mixed value per room, a 3 m ceiling, no smoke
  layer, no flashover beyond the 900 °C cap.
- **The sprinkler check reads the true temperature**, not a sensor. That models a
  heat-activated head, but it is one more place outside the sensors that sees the truth.
- **Evacuation is all or nothing.** One confirmed room empties the whole building.
- **Labels are per room.** Smoke that drifted into the room next door is labelled `none`.
- **History is short:** about 20 simulated minutes per room.
- **Reconnecting is manual.** After BuildSim fails, drawing stays off until you press Reload.
- **One building, one run.** Nothing is saved between restarts.